# Mutual Fund Facts Assistant: Technical Implementation

This notebook contains the complete technical implementation of the Mutual Fund Facts Assistant. Unlike the walkthrough notebook, this version implements all the logic directly in the cells, making it ideal for technical review and development.

## 1. Imports and Configuration
We start by importing the necessary libraries for vector database management, embeddings, keyword search, and LLM generation.

In [ ]:
import chromadb
import re
import time
import os
from sentence_transformers import SentenceTransformer, CrossEncoder
from groq import Groq
from rank_bm25 import BM25Okapi

# Constants
DATABASE_FOLDER = "chroma_db"
COLLECTION_NAME = "quant_mutual_fund"
MODEL_NAME = "all-MiniLM-L6-v2"
NUMBER_OF_RESULTS = 10

SOURCE_URLS = {
    "SID_quant_Small_Cap_Fund_March_2026.md": "https://quantmutual.com/Admin/SIDPdf/SID%20quant%20Small%20Cap%20Fund%20March%202026.pdf",
    "Presentation_for_Quant_Small_Cap_Fund.md": "https://quantmutual.com/Admin/Pdf/quant%20Small%20Cap%20Fund_Presentation.pdf",
    "quant_Small_Cap_Fund.md": "https://quantmutual.com/Admin/Pdf/quant_Small_Cap_Fund.pdf",
    "SID_quant_Large_Cap_Fund_Sep_2025.md": "https://quantmutual.com/Admin/SIDPdf/SID_quant_Large_Cap_Fund_Sep_2025.pdf",
    "quant_Large_Cap_Fund_Presentation.md": "https://quantmutual.com/Admin/Pdf/quant%20Large%20Cap%20Fund%20Presentation.pdf",
    "quant_Large_Cap_Fund.md": "https://quantmutual.com/Admin/Pdf/quant_Large_Cap_Fund.pdf",
    "SID_quant_Flexi_Cap_Fund_September_2025.md": "https://quantmutual.com/Admin/SIDPdf/SID_quant_Flexi_Cap_Fund_September_2025.pdf",
    "quant_Flexi_Cap_Fund_Presentation-_Updated.md": "https://quantmutual.com/Admin/Pdf/quant_Flexi_Cap_Fund_Presentation-%20Updated.pdf",
    "quant_Flexi_Cap_Fund.md": "https://quantmutual.com/Admin/Pdf/quant_Flexi_Cap_Fund.pdf",
    "SID_quant_PSU_Fund_March_2026.md": "https://quantmutual.com/Admin/SIDPdf/SID%20quant%20PSU%20Fund%20March%202026.pdf",
    "quant_PSU_Fund_Presentation.md": "https://quantmutual.com/Admin/Pdf/quant%20PSU%20Fund%20Presentation.pdf",
    "quant_PSU_Fund.md": "https://quantmutual.com/Admin/Pdf/quant_PSU_Fund.pdf",
    "SID_quant_Multi_Cap_Fund_March_2026.md": "https://quantmutual.com/Admin/SIDPdf/SID%20quant%20Multi%20Cap%20Fund%20March%202026.pdf",
    "quant_Multi_Cap_Fund_Presentation.md": "https://quantmutual.com/Admin/Pdf/quant%20Multi%20Cap%20Fund_Presentation.pdf",
    "quant_Multi_Cap_Fund.md": "https://quantmutual.com/Admin/Pdf/quant_Multi_Cap_Fund.pdf",
    "quant_Factsheet_-_September_2026.md": "https://quantmutual.com/Admin/Factsheet/quant_Factsheet_-_September_2026.pdf",
    "Definitions_and_Interpretation.md": "https://quantmutual.com/Pdf/Definitions_and_Interpretation.pdf",
    "Penalities.md": "https://quantmutual.com/Pdf/Penalities.pdf",
    "Principles_of_Incentive_Structure_for_Market_Makers.md": "https://quantmutual.com/Admin/SIDPdf/Principles_of_Incentive_Structure_for_Market_Makers.pdf",
    "Statement_of_Additional_Information.md": "https://quantmutual.com/Admin/SIDPdf/Statement_of_Additional_Information.pdf",
    "1727242783639.md": "https://www.sebi.gov.in/sebi_data/faqfiles/sep-2024/1727242783639.pdf",
    "QMF_Schemes.md": "https://quantmutual.com/Admin/Pdf/QMF_Schemes.pdf",
    "quant_TER.md": "https://www.quantmutual.com/Total-Expense-Ratio",
}

SOURCE_DATES = {
    "SID_quant_Small_Cap_Fund_March_2026.md": "March 2026",
    "SID_quant_Large_Cap_Fund_Sep_2025.md": "September 2025",
    "SID_quant_Flexi_Cap_Fund_September_2025.md": "September 2025",
    "SID_quant_PSU_Fund_March_2026.md": "March 2026",
    "SID_quant_Multi_Cap_Fund_March_2026.md": "March 2026",
    "quant_Factsheet_-_September_2026.md": "September 2026",
    "1727242783639.md": "September 2024",
}

## 2. System Initialization
We initialize the embedding model and the ChromaDB persistent client to access the existing knowledge base.

In [ ]:
print("Loading RAG system...")

embedding_model = SentenceTransformer(MODEL_NAME)

chroma_client = chromadb.PersistentClient(path=DATABASE_FOLDER)
collection = chroma_client.get_collection(name=COLLECTION_NAME)

print(f"Ready! Database contains {collection.count()} chunks.")

## 3. Keyword Retrieval (BM25) Setup
To implement hybrid search, we load all documents from ChromaDB and initialize a BM25 index for keyword-based retrieval.

In [ ]:
all_chunks = collection.get(include=["documents", "metadatas"])
bm25_documents = all_chunks["documents"]
bm25_metadatas = all_chunks["metadatas"]

def tokenize(text):
    return re.findall(r"[a-z0-9]+", text.lower())

tokenized_documents = [tokenize(doc) for doc in bm25_documents]
bm25 = BM25Okapi(tokenized_documents)
print("BM25 Index initialized.")

## 4. Intent Detection and Guardrails
We use a set of example phrases to detect the user's intent via semantic similarity and implement strict guardrails to prevent investment advice and PII leakage.

In [ ]:
INTENT_EXAMPLES = {
    "exit_load": ["What is the exit load?", "Will I be charged if I withdraw my investment early?", "Is there a penalty if I redeem my units?", "Does the fund deduct anything when I cash out?"],
    "minimum_investment": ["What is the minimum investment amount?", "How much money do I need to start investing?", "What is the minimum lump sum investment?", "How much is required for my first one-time investment?"],
    "sip": ["What is the minimum SIP amount?", "How little can I invest every month?", "What is the smallest recurring investment allowed?", "What is the minimum SIP instalment?"],
    "fund_manager": ["Who is the fund manager?", "Who manages the fund?", "Who handles the investment portfolio?", "Who is responsible for managing the scheme?"],
    "ter": ["What is the expense ratio?", "What is the TER?", "What are the charges for Direct and Regular plans?", "What are the ongoing annual fund charges?"],
    "investment_objective": ["What is the investment objective?", "What does the fund aim to achieve?", "What outcome is the fund designed to pursue?", "What is the purpose of the investment strategy?"],
    "asset_allocation": ["What is the asset allocation?", "Where does the fund invest its money?", "What percentage can be invested in each asset class?", "What are the permitted investment ranges?"],
    "redemption_timeline": ["When will I receive my redemption proceeds?", "How long does redemption payment take?", "How soon will I receive my money after redeeming units?", "When will the payout reach me after I sell my units?"],
    "riskometer": ["What is the Riskometer level of the fund?", "What is the risk level of the scheme?", "How risky is this mutual fund according to the Riskometer?", "What does the scheme Riskometer say?"],
    "benchmark": ["What is the benchmark of the fund?", "Which benchmark index does the scheme use?", "What index is the fund benchmarked against?", "What is the scheme benchmark?"],
    "account_statement": ["How can I get my account statement?", "How do I request an account statement?", "When will I receive my account statement?", "How can I get my Consolidated Account Statement?", "How do I get my CAS?"],
    "capital_gains": ["What are the capital gains tax rules?", "What tax applies to capital gains from this fund?", "What is the long-term capital gains tax?", "What is the short-term capital gains tax?", "How are capital gains from this mutual fund taxed?"]
}

intent_names = []
intent_texts = []
for intent_name, examples in INTENT_EXAMPLES.items():
    for example in examples:
        intent_names.append(intent_name)
        intent_texts.append(example)

intent_embeddings = embedding_model.encode(intent_texts, normalize_embeddings=True)

def apply_guardrails(question):
    question_lower = question.lower()
    advice_phrases = ["should i invest", "should i buy", "should i sell", "should i redeem", "should i hold", "is it a good investment", "is this a good investment", "which fund should i buy", "which fund should i invest", "which fund is best", "recommend"]
    if any(phrase in question_lower for phrase in advice_phrases) or "a good investment" in question_lower or ("which" in question_lower and "best" in question_lower):
        return "I can provide factual information about mutual funds, but I can't recommend whether you should buy, sell, hold, redeem, or invest in a particular fund.\nInvestor education: https://www.sebi.gov.in/sebi_data/faqfiles/sep-2024/1727242783639.pdf\nLast updated from sources: September 2024"
    
    performance_phrases = ["compare the returns", "expected return", "expected returns", "expect from", "how much return", "how much returns", "past return", "past returns", "historical return", "historical returns", "fund performance", "performance of", "performed", "cagr", "xirr"]
    if any(phrase in question_lower for phrase in performance_phrases) or ("returns" in question_lower and "highest" in question_lower):
        return "I don't provide, predict, calculate, or compare mutual fund returns or performance. For official performance information, please refer to the Quant Mutual Fund factsheet:\nhttps://quantmutual.com/Admin/Factsheet/quant_Factsheet_-_September_2026.pdf\nLast updated from sources: September 2026"
    
    pii_patterns = [r"\b[A-Z]{5}[0-9]{4}[A-Z]\b", r"\b\d{4}\s?\d{4}\s?\d{4}\b", r"\b\d{10}\b", r"\b[\w\.-]+@[\w\.-]+\.\w+\b"]
    pii_phrases = ["my pan is", "my aadhaar", "my aadhar", "my account number", "my otp"]
    if any(re.search(pattern, question, re.IGNORECASE) for pattern in pii_patterns) or any(phrase in question_lower for phrase in pii_phrases):
        return "Please do not share personal or sensitive information such as PAN, Aadhaar, account numbers, OTPs, email addresses, or phone numbers. I can only provide general factual information about mutual funds."
    
    return None

print("Guardrails and Intent detection initialized.")

## 5. The RAG Pipeline (`ask_rag`)
The main function combines semantic retrieval, keyword retrieval, optional reranking, and Groq LLM generation.

In [ ]:
USE_RERANKER = os.getenv("USE_RERANKER", "true").lower() == "true"
reranker = CrossEncoder("cross-encoder/ms-marco-MiniL-L-6-v2") if USE_RERANKER else None

def add_source_citation(answer, metadata):
    source_file = metadata.get("source_file", "")
    source_url = SOURCE_URLS.get(source_file)
    source_date = SOURCE_DATES.get(source_file, "Date not specified in source")
    if source_url and source_date: return f"{answer}\nSource: {source_url}\nLast updated from sources: {source_date}"
    if source_url: return f"{answer}\nSource: {source_url}"
    return answer

def ask_rag(question):
    # 1. Guardrails
    guardrail_response = apply_guardrails(question)
    if guardrail_response: return guardrail_response
    
    question_lower = question.lower()
    
    # 2. Fund Detection
    selected_fund = None
    if "small cap" in question_lower: selected_fund = "Quant Small Cap Fund"
    elif "large cap" in question_lower: selected_fund = "Quant Large Cap Fund"
    elif "flexi cap" in question_lower: selected_fund = "Quant Flexi Cap Fund"
    elif "multi cap" in question_lower: selected_fund = "Quant Multi Cap Fund"
    elif "psu" in question_lower: selected_fund = "Quant PSU Fund"
    
    # 3. Intent Detection
    q_emb = embedding_model.encode(question, normalize_embeddings=True)
    intent_scores = {name: max([float(q_emb @ intent_embeddings[i]) for i, n in enumerate(intent_names) if n == name]) for name in INTENT_EXAMPLES}
    semantic_intent = max(intent_scores, key=intent_scores.get)
    
    # 4. Hybrid Retrieval
    # BM25
    tokenized_query = tokenize(question_lower)
    eligible_indices = [i for i, m in enumerate(bm25_metadatas) if selected_fund is None or m.get("fund_name") == selected_fund]
    fund_bm25 = BM25Okapi([tokenized_documents[i] for i in eligible_indices])
    fund_scores = fund_bm25.get_scores(tokenized_query)
    bm25_results = sorted([{"document": bm25_documents[idx], "metadata": bm25_metadatas[idx], "score": score} for idx, score in zip(eligible_indices, fund_scores)], key=lambda x: x["score"], reverse=True)[:40]
    
    # Semantic
    results = collection.query(query_texts=[question], n_results=10, where={"$and": [{"fund_name": selected_fund}] if selected_fund else {}})
    
    # Combined & Rerank
    combined_docs = results["documents"][0] + [r["document"] for r in bm25_results]
    combined_meta = results["metadatas"][0] + [r["metadata"] for r in bm25_results]
    
    if reranker:
        scores = reranker.predict([[question, doc] for doc in combined_docs])
        reranked = sorted(zip(scores, combined_docs, combined_meta), key=lambda x: x[0], reverse=True)
        final_docs = [x[1] for x in reranked[:6]]
        final_meta = [x[2] for x in reranked[:6]]
    else:
        final_docs, final_meta = combined_docs[:6], combined_meta[:6]
    
    # 5. Generation
    context = "\n\n".join([f"SOURCE {i+1}\n{doc}" for i, doc in enumerate(final_docs)])
    prompt = f"Answer the USER QUESTION using ONLY the RETRIEVED CONTEXT.\n\nQUESTION: {question}\n\nCONTEXT: {context}"
    
    client = Groq()
    response = client.chat.completions.create(model="openai/gpt-oss-20b", messages=[{"role": "user", "content": prompt}], temperature=0)
    
    return add_source_citation(response.choices[0].message.content, final_meta[0])

print("RAG Pipeline ready.")

## 6. Demonstration
Test the full pipeline with factual questions.

In [ ]:
if not os.getenv("GROQ_API_KEY"):
    print("Set GROQ_API_KEY env var first!")
else:
    print(ask_rag("What is the minimum SIP for Quant Small Cap Fund?"))